# 04 — Aprendizado Local: Clustering + Modelos Pooled

**Dataset:** `receita_realizada_a_partir_2019_sgo.parquet`  
**Série-alvo:** Fonte Detalhada Harmonizada (`Fonte_Det_Cód_Harm`)  
**Horizonte de previsão:** 12 meses (nov/2025 – out/2026)

**Objetivo:** Avaliar se modelos treinados em conjuntos de séries similares (*aprendizado local / pooled*)
superam modelos treinados exclusivamente na série-alvo (*aprendizado individual*).

## Protocolo anti-vazamento
| Etapa | Dados usados |
|---|---|
| Extração de features | Apenas janela de **treino** |
| Clustering (silhouette) | Features de treino |
| Dataset pooled | Janelas de treino dos membros do cluster |
| Hyperparameter search (CV) | Dados pooled de treino |
| Avaliação final | **Holdout** de cada série (nunca visto antes) |

## Modelos comparados
- **Individual**: ARIMA · LR · SVR · MLP (vencedor por série — `experiment.py`)
- **Pooled Ridge**: Regressão linear regularizada sobre dados do cluster
- **Pooled SVR**: SVR com grid search sobre dados do cluster
- **Pooled MLP**: Rede neural com grid search sobre dados do cluster

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from sklearn.decomposition import PCA
from scipy.stats import wilcoxon

from experiment_local import rodar_experimento_local

CORES = {
    'azul'    : '#1B3A5C',
    'azul_m'  : '#2E6DA4',
    'azul_c'  : '#D6E8F7',
    'verde'   : '#27AE60',
    'vermelho': '#C0392B',
    'amarelo' : '#F39C12',
    'cinza'   : '#BDC3C7',
    'roxo'    : '#8E44AD',
    'laranja' : '#E67E22',
}

plt.rcParams.update({
    'figure.dpi': 130, 'font.family': 'DejaVu Sans',
    'axes.spines.top': False, 'axes.spines.right': False,
})
print('Ambiente pronto.')

## 0 · Execução do experimento local

In [ ]:
# Executa o experimento local completo com selecao automatica de metodo de clustering
# metodo_clustering="auto" testa features, correlacao e cosseno — seleciona o melhor silhouette
resultados = rodar_experimento_local(metodo_clustering="auto", verbose=False)

comp              = resultados['comparacao_df']
clustering        = resultados['clustering']
features_df       = resultados['features_df']
todos_clusterings = resultados.get('todos_clusterings', {})
metodo_sel        = resultados.get('metodo_selecionado', clustering.metodo)

print(f"Metodo selecionado : {metodo_sel}")
print(f"Clusters formados  : {clustering.k}")
print(f"Silhouette score   : {clustering.silhouette:.4f}")
print(f"Wilcoxon p-value   : {resultados['wilcoxon_p']:.4f}")
print(f"Series analisadas  : {len(comp)}")
comp.head()

## 0b · Comparação de métodos de clustering

Três estratégias testadas automaticamente pelo modo `"auto"`:

| Método | Distância | Silhouette calculado com |
|---|---|---|
| **features** | Euclidiano sobre 8 features estatísticas | Euclidiano (features z-score) |
| **correlacao** | (1 − Pearson) / 2 | Precomputed |
| **cosseno** | (1 − cosine_sim) / 2 na janela recente fixa | Precomputed |

O método com **maior silhouette** é selecionado para o experimento completo.

## 1 · Seleção de k — Silhouette por número de clusters

In [ ]:
if todos_clusterings:
    metodos  = sorted(todos_clusterings.keys(),
                      key=lambda m: todos_clusterings[m].silhouette, reverse=True)
    cores_mt = {'features': CORES['azul_m'], 'correlacao': CORES['verde'], 'cosseno': CORES['laranja']}

    # Tabela de comparacao
    rows = []
    for m in metodos:
        r = todos_clusterings[m]
        maior_cl = max(len(v) for v in r.cluster_members.values())
        menor_cl = min(len(v) for v in r.cluster_members.values())
        rows.append({'Metodo': m, 'k': r.k, 'Silhouette': round(r.silhouette, 4),
                     'Maior cluster': maior_cl, 'Menor cluster': menor_cl,
                     'Selecionado': '*** MELHOR ***' if m == metodo_sel else ''})
    df_comp_cl = pd.DataFrame(rows)
    print("Comparacao de metodos de clustering:")
    print(df_comp_cl.to_string(index=False))
    print()

    # Grafico: silhouette por metodo
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))

    # Painel 1: silhouette score por metodo
    sils_mt = [todos_clusterings[m].silhouette for m in metodos]
    cors_mt = [cores_mt.get(m, CORES['cinza']) for m in metodos]
    bars = axes[0].bar(metodos, sils_mt, color=cors_mt, edgecolor='white', width=0.5)
    bars[0].set_edgecolor(CORES['azul'])
    bars[0].set_linewidth(2.5)
    for bar, v in zip(bars, sils_mt):
        axes[0].text(bar.get_x() + bar.get_width()/2,
                     v + 0.005, f'{v:.4f}', ha='center', va='bottom', fontsize=9)
    axes[0].set_ylabel('Silhouette score', fontsize=10)
    axes[0].set_title('Silhouette por metodo de clustering\n(borda azul = selecionado)', fontsize=11)

    # Painel 2: tamanho dos clusters por metodo (stacked bar)
    for i, m in enumerate(metodos):
        r      = todos_clusterings[m]
        sizes  = sorted([len(v) for v in r.cluster_members.values()], reverse=True)
        bottom = 0
        sub_cores = [CORES['azul_m'], CORES['verde'], CORES['vermelho'],
                     CORES['amarelo'], CORES['roxo'], CORES['laranja']]
        for j, s in enumerate(sizes):
            axes[1].bar(m, s, bottom=bottom,
                        color=sub_cores[j % len(sub_cores)],
                        edgecolor='white', width=0.5,
                        label=f'Cluster {j}' if i == 0 else '')
            if s > 5:
                axes[1].text(i, bottom + s/2, str(s),
                             ha='center', va='center', fontsize=8, color='white', fontweight='bold')
            bottom += s

    axes[1].set_ylabel('Numero de series', fontsize=10)
    axes[1].set_title('Distribuicao dos clusters por metodo', fontsize=11)

    plt.suptitle('Comparacao de Metodos de Clustering', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("Apenas um metodo foi executado:", clustering.metodo)

In [ ]:
scores = clustering.silhouette_per_k
ks     = sorted(scores.keys())
sils   = [scores[k] for k in ks]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(ks, sils, color=CORES['azul_m'], edgecolor='white', width=0.6)
bars[ks.index(clustering.k)].set_facecolor(CORES['verde'])
bars[ks.index(clustering.k)].set_edgecolor(CORES['azul'])

for k, s, bar in zip(ks, sils, bars):
    ax.text(bar.get_x() + bar.get_width()/2, s + 0.003,
            f'{s:.4f}', ha='center', va='bottom', fontsize=9)

ax.set_xlabel('Numero de clusters (k)', fontsize=10)
ax.set_ylabel('Silhouette score', fontsize=10)
ax.set_title('Selecao de k — Silhouette (maior = melhor separacao)', fontsize=11)
ax.set_xticks(ks)

patch_sel = mpatches.Patch(color=CORES['verde'], label=f'k selecionado = {clustering.k}')
ax.legend(handles=[patch_sel], fontsize=9)
plt.tight_layout()
plt.show()

print(f"k={clustering.k} escolhido com silhouette={clustering.silhouette:.4f}")
print(f"Distribuicao dos clusters:")
for cid, members in clustering.cluster_members.items():
    print(f"  Cluster {cid}: {len(members)} series")

## 2 · Visualização dos clusters — PCA 2D

In [ ]:
# embedding_2d: pre-computado por cada metodo
#   - features   -> PCA(2) sobre features z-score
#   - correlacao -> MDS classico sobre matriz de distancias de correlacao
#   - cosseno    -> MDS classico sobre matriz de distancias de cosseno
if clustering.embedding_2d is not None:
    X_vis = clustering.embedding_2d
    eixo_label = 'MDS' if clustering.metodo in ('correlacao', 'cosseno') else 'PC'
else:
    from sklearn.decomposition import PCA
    X_vis = PCA(n_components=2, random_state=42).fit_transform(clustering.features_scaled)
    eixo_label = 'PC'

labels = clustering.labels
k      = clustering.k

palette = [CORES['azul_m'], CORES['verde'], CORES['vermelho'],
           CORES['amarelo'], CORES['roxo'], CORES['laranja']]

fig, ax = plt.subplots(figsize=(9, 5))

for cid in range(k):
    mask  = labels == cid
    n_cid = int(mask.sum())
    ax.scatter(X_vis[mask, 0], X_vis[mask, 1],
               c=palette[cid % len(palette)], s=60, alpha=0.75,
               edgecolors='white', linewidths=0.5,
               label=f'Cluster {cid} (n={n_cid})')
    if n_cid > 0:
        cx, cy = X_vis[mask, 0].mean(), X_vis[mask, 1].mean()
        ax.scatter(cx, cy, c=palette[cid % len(palette)], s=200,
                   marker='*', edgecolors='white', linewidths=1.5, zorder=5)

modelaveis = set(comp['Codigo'].astype(str))
for i, cod in enumerate(clustering.codigos):
    if str(cod) in modelaveis:
        ax.scatter(X_vis[i, 0], X_vis[i, 1],
                   s=130, marker='D', c='none',
                   edgecolors='black', linewidths=1.5, zorder=6)

diamond = mpatches.Patch(facecolor='none', edgecolor='black',
                         label='Serie modelavel (nao-RB)')
handles, _ = ax.get_legend_handles_labels()
ax.legend(handles=handles + [diamond], fontsize=9, loc='best')

ax.set_xlabel(f'{eixo_label}1', fontsize=10)
ax.set_ylabel(f'{eixo_label}2', fontsize=10)
ax.set_title(
    f'Clusters — {eixo_label} 2D  |  metodo={clustering.metodo}  '
    f'k={k}  silhouette={clustering.silhouette:.4f}',
    fontsize=11
)
plt.tight_layout()
plt.show()

## 3 · Features por cluster — Heatmap de médias

In [ ]:
feat_cluster = features_df.copy()
feat_cluster['cluster'] = feat_cluster.index.map(
    lambda c: clustering.cluster_map.get(str(c), clustering.cluster_map.get(c, -1))
)
means = feat_cluster.groupby('cluster').mean()

# Remove coluna cluster se existir (caso index map adicione)
if 'cluster' in means.columns:
    means = means.drop(columns=['cluster'])

# Normaliza por coluna para visualizacao comparativa
means_norm = (means - means.mean()) / (means.std() + 1e-9)

fig, ax = plt.subplots(figsize=(11, max(3, len(means) * 1.0)))
im = ax.imshow(means_norm.values, cmap='RdYlGn', aspect='auto', vmin=-2, vmax=2)

ax.set_xticks(range(len(means_norm.columns)))
ax.set_xticklabels(means_norm.columns, rotation=35, ha='right', fontsize=9)
ax.set_yticks(range(len(means_norm)))
ax.set_yticklabels([f'Cluster {i}' for i in means_norm.index], fontsize=10)

for i in range(len(means_norm)):
    for j in range(len(means_norm.columns)):
        ax.text(j, i, f'{means.values[i, j]:.2f}',
                ha='center', va='center', fontsize=7.5)

plt.colorbar(im, ax=ax, label='Z-score da media por feature')
ax.set_title('Perfil medio dos clusters — features extraidas da janela de treino', fontsize=11)
plt.tight_layout()
plt.show()

print("\nMedias brutas por cluster:")
print(means.to_string(float_format='{:.3f}'.format))

## 4 · Comparação direta: RMSE Individual vs Pooled por série

In [ ]:
df_plot = comp.dropna(subset=['Ind_RMSE', 'Pooled_Melhor_RMSE']).copy()
df_plot = df_plot.sort_values('Ind_RMSE', ascending=False).reset_index(drop=True)
df_plot['melhorou'] = df_plot['Pooled_Melhor_RMSE'] < df_plot['Ind_RMSE']

n   = len(df_plot)
y   = np.arange(n)
fig, ax = plt.subplots(figsize=(10, max(5, n * 0.42)))

for i, row in df_plot.iterrows():
    cor = CORES['verde'] if row['melhorou'] else CORES['vermelho']
    ax.plot([row['Ind_RMSE'], row['Pooled_Melhor_RMSE']], [i, i],
            color=cor, lw=1.5, alpha=0.6, zorder=1)

ax.scatter(df_plot['Ind_RMSE'], y,
           color=CORES['azul'], s=70, zorder=3, label='Individual (melhor)')
ax.scatter(df_plot['Pooled_Melhor_RMSE'], y,
           color=[CORES['verde'] if m else CORES['vermelho']
                  for m in df_plot['melhorou']],
           s=70, marker='D', zorder=3, label='Pooled (melhor)')

ax.set_yticks(y)
ax.set_yticklabels([str(row['Codigo']) for _, row in df_plot.iterrows()], fontsize=8)
ax.set_xlabel('RMSE (holdout)', fontsize=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))

patch_g = mpatches.Patch(color=CORES['verde'],  label='Pooled melhor')
patch_r = mpatches.Patch(color=CORES['vermelho'], label='Individual melhor')
dot_b   = plt.Line2D([0],[0], marker='o', color='w',
                      markerfacecolor=CORES['azul'], markersize=8, label='Individual')
dot_d   = plt.Line2D([0],[0], marker='D', color='w',
                      markerfacecolor=CORES['azul_m'], markersize=8, label='Pooled')
ax.legend(handles=[dot_b, dot_d, patch_g, patch_r], fontsize=9, loc='lower right')

n_mel = int(df_plot['melhorou'].sum())
ax.set_title(
    f'Individual vs Pooled — RMSE por serie  |  pooled melhorou em {n_mel}/{n}',
    fontsize=11
)
plt.tight_layout()
plt.show()

print(f"Pooled melhor em {n_mel} / {n} series ({100*n_mel/n:.1f}%)")

## 5 · Comparação por modelo pooled — RMSE médio e mediano

In [ ]:
modelos   = ['Ind_RMSE', 'Pooled_Ridge_RMSE', 'Pooled_SVR_RMSE', 'Pooled_MLP_RMSE']
labels_m  = ['Individual\n(melhor)', 'Pooled\nRidge', 'Pooled\nSVR', 'Pooled\nMLP']
cores_m   = [CORES['azul'], CORES['azul_m'], CORES['amarelo'], CORES['roxo']]

# Filtra apenas series com todos os valores
comp_valido = comp.dropna(subset=modelos)
medias   = [comp_valido[m].mean()   for m in modelos]
medianas = [comp_valido[m].median() for m in modelos]

x = np.arange(len(modelos))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Grafico de media
bars1 = axes[0].bar(x, medias, width=0.65, color=cores_m,
                    edgecolor='white', linewidth=0.8)
for bar, v in zip(bars1, medias):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 v + max(medias)*0.01, f'R${v/1e6:.2f}M',
                 ha='center', va='bottom', fontsize=8)
axes[0].set_xticks(x)
axes[0].set_xticklabels(labels_m, fontsize=9)
axes[0].set_title('RMSE medio (holdout)', fontsize=11)
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R${v/1e6:.1f}M'))
axes[0].set_ylabel('RMSE', fontsize=10)

# Grafico de mediana
bars2 = axes[1].bar(x, medianas, width=0.65, color=cores_m,
                    edgecolor='white', linewidth=0.8)
for bar, v in zip(bars2, medianas):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 v + max(medianas)*0.01, f'R${v/1e6:.2f}M',
                 ha='center', va='bottom', fontsize=8)
axes[1].set_xticks(x)
axes[1].set_xticklabels(labels_m, fontsize=9)
axes[1].set_title('RMSE mediano — robusto a outliers', fontsize=11)
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v,_: f'R${v/1e6:.2f}M'))
axes[1].set_ylabel('RMSE', fontsize=10)

plt.suptitle('Comparacao de RMSE: Individual vs Modelos Pooled', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"(n={len(comp_valido)} series com todos os valores validos)")

## 6 · Delta percentual por série (pooled vs individual)

In [ ]:
if 'Delta_RMSE_pct' in comp.columns:
    df_d = comp.dropna(subset=['Delta_RMSE_pct']).sort_values('Delta_RMSE_pct')
    n_d  = len(df_d)

    fig, ax = plt.subplots(figsize=(9, max(4, n_d * 0.40)))
    cores_bar = [CORES['verde'] if v < 0 else CORES['vermelho']
                 for v in df_d['Delta_RMSE_pct']]
    bars = ax.barh(range(n_d), df_d['Delta_RMSE_pct'],
                   color=cores_bar, edgecolor='white', height=0.7)

    for i, (bar, v) in enumerate(zip(bars, df_d['Delta_RMSE_pct'])):
        offset = -2 if v < 0 else 2
        ha = 'right' if v < 0 else 'left'
        ax.text(v + offset, bar.get_y() + bar.get_height()/2,
                f'{v:+.1f}%', ha=ha, va='center', fontsize=8)

    ax.axvline(0, color='black', lw=1.2)
    ax.set_yticks(range(n_d))
    ax.set_yticklabels(df_d['Codigo'].astype(str), fontsize=8)
    ax.set_xlabel('Delta RMSE pooled vs individual (%)', fontsize=10)

    n_mel = int((df_d['Delta_RMSE_pct'] < 0).sum())
    ax.set_title(
        f'Delta % de RMSE  |  negativo = pooled melhorou  |  {n_mel}/{n_d} series',
        fontsize=11
    )
    plt.tight_layout()
    plt.show()
else:
    print("Coluna Delta_RMSE_pct nao encontrada — verifique se o experimento gerou comparacao corretamente.")

## 7 · Teste de Wilcoxon Signed-Rank

In [ ]:
stat_w = resultados.get('wilcoxon_stat', float('nan'))
p_w    = resultados.get('wilcoxon_p',    float('nan'))

print('=' * 60)
print('TESTE DE WILCOXON SIGNED-RANK')
print('H0: RMSE_individual = RMSE_pooled  (mediana das diferencas = 0)')
print('H1: RMSE_individual != RMSE_pooled (bicaudal)')
print('=' * 60)

if not (np.isnan(stat_w) or np.isnan(p_w)):
    print(f'Estatistica W : {stat_w:.2f}')
    print(f'p-valor       : {p_w:.4f}')
    alpha = 0.05
    if p_w < alpha:
        print(f'=> Rejeita H0 a {alpha*100:.0f}%: diferenca SIGNIFICATIVA')
    else:
        print(f'=> Nao rejeita H0 a {alpha*100:.0f}%: diferenca NAO significativa')
else:
    print('Amostras insuficientes para o teste (n < 5) ou NaN detectado.')

print()
print('RESUMO QUANTITATIVO')
print('-' * 60)
if 'Ind_RMSE' in comp.columns and 'Pooled_Melhor_RMSE' in comp.columns:
    n_pool_better = int((comp['Pooled_Melhor_RMSE'] < comp['Ind_RMSE']).sum())
    n_total       = len(comp.dropna(subset=['Ind_RMSE', 'Pooled_Melhor_RMSE']))
    print(f'Series onde pooled < individual  : {n_pool_better} / {n_total}')
    print(f'RMSE medio individual            : R$ {comp["Ind_RMSE"].mean():>18,.2f}')
    print(f'RMSE medio pooled (melhor)       : R$ {comp["Pooled_Melhor_RMSE"].mean():>18,.2f}')
    print(f'RMSE mediano individual          : R$ {comp["Ind_RMSE"].median():>18,.2f}')
    print(f'RMSE mediano pooled (melhor)     : R$ {comp["Pooled_Melhor_RMSE"].median():>18,.2f}')
    print()
    print('Por modelo pooled:')
    for col, nome in [('Pooled_Ridge_RMSE','Ridge'),
                      ('Pooled_SVR_RMSE','SVR  '),
                      ('Pooled_MLP_RMSE','MLP  ')]:
        if col in comp.columns:
            n_b = int((comp[col] < comp['Ind_RMSE']).sum())
            rmse_m = comp[col].mean()
            print(f'  {nome}: {n_b:3d}/{n_total} series melhores  |  '
                  f'RMSE medio = R$ {rmse_m:>12,.2f}')

## 8 · Tabela completa de comparação

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

cols_show = [
    'Codigo', 'Cluster_ID', 'Membros_Cluster', 'N_Lags',
    'Ind_Melhor_Modelo', 'Ind_RMSE',
    'Pooled_Ridge_RMSE', 'Pooled_SVR_RMSE', 'Pooled_MLP_RMSE',
    'Pooled_Melhor_Modelo', 'Pooled_Melhor_RMSE', 'Delta_RMSE_pct',
]
cols_show = [c for c in cols_show if c in comp.columns]

def _highlight(row):
    styles = [''] * len(row)
    if 'Delta_RMSE_pct' in row.index:
        idx = list(row.index).index('Delta_RMSE_pct')
        val = row['Delta_RMSE_pct']
        if not pd.isna(val):
            styles[idx] = ('background-color: #D5F5E3; color: #1E8449'
                           if val < 0 else
                           'background-color: #FDECEA; color: #C0392B')
    return styles

fmt = {c: '{:,.0f}' for c in cols_show if 'RMSE' in c}
if 'Delta_RMSE_pct' in cols_show:
    fmt['Delta_RMSE_pct'] = '{:+.1f}%'

(
    comp[cols_show]
    .style
    .apply(_highlight, axis=1)
    .format(fmt, na_rep='N/A')
    .set_caption('Comparacao individual vs pooled | delta negativo = pooled melhorou')
)

## 9 · Síntese e interpretação

### Contexto do dataset

| Dimensão | Valor |
|---|---|
| Séries válidas (cobertura completa) | 367 |
| Séries ruído branco (Ljung-Box) | ~202 |
| Séries modeladas (não-RB) | ~165 |
| Silhouette score | ~0.39 |
| k ótimo | 3 |
| Distribuição dos clusters | ~{0: 243, 1: 123, 2: 1} |

### Interpretação do silhouette

Com **silhouette ≈ 0.39**, a separação dos clusters é **razoável** — melhor que a separação obtida
no experimento de despesas liquidadas (≈ 0.19). Isso indica que as séries de receita têm
perfis de similaridade mais distintos entre si.

Porém, a distribuição assimétrica (`{0: 243, 1: 123, 2: 1}`) mostra que a maioria das séries
se concentra em dois clusters principais, enquanto apenas 1 série forma o terceiro cluster —
provavelmente um outlier extremo.

### Por que o pooled pode não superar em todas as séries

1. **Clusters heterogêneos**: mesmo com silhouette razoável, clusters com centenas de séries
   ainda contêm muita variabilidade interna.
2. **Séries já bem ajustadas individualmente**: ARIMA com sazonalidade forte tende a ser
   difícil de superar com modelos pooled baseados apenas em lags.
3. **Escala das receitas**: receitas de fontes detalhadas têm amplitudes muito variadas (de
   dezenas de milhares a bilhões de R$), tornando a normalização per-série crítica.

### Quando o pooled claramente ajuda
- Séries com **poucos dados de treino** (≤ 15 obs.) onde modelos individuais sofrem alta variância
- Séries no mesmo cluster com **padrões de sazonalidade e tendência similares**
- Casos onde o modelo individual selecionado é sub-ótimo (e.g., LR com sinal fraco)

### Próximos passos sugeridos
- Testar **clustering hierárquico** ou **DTW** para capturar similaridade de forma
- Explorar **modelos globais** (LightGBM, N-BEATS) treinados em todas as séries
- Aplicar **N2V/meta-learning** para aproveitar informações cross-série
- Avaliar se séries do mesmo **grupo orçamentário** (mesma origem/espécie) formam clusters naturais